## 模块：模块化与异常处理 —— 让代码更健壮、更有序

**驱动问题**：我们要开发一个调用大模型API的工具。代码会越来越多，需要拆分成多个文件来管理；网络请求可能失败，需要提前考虑出错处理。本节聚焦两大核心技能：**用模块和包组织代码**，以及**用异常处理让程序在出错时“优雅降级”**。最后简要了解正则表达式，以便进行简单的文本处理。

---

### 1. 模块与包 —— 组织你的代码

#### 1.1 核心概念
- **模块**：一个 `.py` 文件就是一个模块，用来封装相关的函数、类和变量。
- **包**：一个包含 `__init__.py` 文件的文件夹，用来组织多个模块。
- **`__name__` 变量**：当Python文件被直接运行时，`__name__` 的值是 `"__main__"`。这让我们的代码既可以作为脚本运行，也可以作为模块被导入。

#### 1.2 三种导入方式
| 语法                      | 说明                   | 适用场景                            |
| ------------------------- | ---------------------- | ----------------------------------- |
| `import module`           | 导入整个模块           | 避免命名冲突，调用时需带模块名      |
| `import module as alias`  | 导入并起别名           | 惯用缩写（如 `import numpy as np`） |
| `from module import name` | 导入特定函数/类        | 简化调用，但可能导致命名冲突        |
| `from module import *`    | 导入所有（**不推荐**） | 污染命名空间，难以调试              |

In [ ]:
# 1. 导入整个模块（最清晰，推荐）
import utils
utils.load_config()
print(utils.count)
ic1 = utils.IceCola("冰可乐", 20)
ic1.speak()

# 2. 从模块中导入特定功能（常用）
from utils import load_config
load_config()
from utils import IceCola

# 3. 给模块或函数起别名
import utils as us
from utils import load_config as lc
lc()

#### 1.3 `if __name__ == "__main__":` 的作用
这行代码是Python脚本的常见入口，它的作用是让`if`代码块里的内容只在**直接运行该文件时**才执行，当文件作为模块被导入时不执行。

In [ ]:
# utils.py
def helper():
    print("我是工具函数")

if __name__ == "__main__":
    print("utils.py 正在被直接运行")
    helper()

#### 1.4 解决模块找不到的问题：动态添加路径

当项目结构变复杂时，子目录中的脚本可能无法导入根目录下的模块。**解决方案**：在入口脚本开头动态将项目根目录加入 `sys.path`。

In [ ]:
import sys
import os

# 获取项目根目录（假设当前脚本在 src/ 子目录下）
current_dir = os.path.dirname(os.path.abspath(__file__))
project_root = os.path.dirname(current_dir)
sys.path.insert(0, project_root)

# 现在可以导入根目录下的自定义包了
# from utils import helper

**专业提示**：更规范的做法是将项目组织为可安装的包（使用 `setup.py` 或 `pyproject.toml`），以开发模式安装（`pip install -e .`），从而避免手动修改 `sys.path`。

---

### 2. 常见标准库的使用

在大模型开发中，API的请求和响应通常是 JSON 格式。`json` 库是处理这类数据的标准工具。

In [ ]:
import json

# Python 字典 -> JSON 字符串
data = {"model": "gpt-4", "temperature": 0.7}
json_str = json.dumps(data, ensure_ascii=False, indent=2)
print(json_str)

# JSON 字符串 -> Python 字典
loaded_data = json.loads(json_str)

# 读写 JSON 文件
with open("config.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

with open("config.json", "r", encoding="utf-8") as f:
    config = json.load(f)
    print(config)

时间的处理操作

In [ ]:
from datetime import datetime, timedelta

now = datetime.now()
print(now.strftime("%Y-%m-%d %H:%M:%S"))

# 解析字符串为时间
dt = datetime.strptime("2025-03-15", "%Y-%m-%d")
print(dt)

# 时间加减
tomorrow = now + timedelta(days=1)
print(tomorrow)

用于HTTP/HTTPS请求的库

In [ ]:
pip install requests

In [ ]:
import requests

def fetch_weather(city):
    """调用公共 API 获取天气（示例使用 wttr.in）"""
    url = f"https://wttr.in/{city}?format=j1"
    resp = requests.get(url, timeout=5)
    if resp.status_code == 200:  # 如果状态码是 200，成功
        data = resp.json()
        current = data["current_condition"][0]
        temp = current["temp_C"]
        desc = current["weatherDesc"][0]["value"]
        return f"{city}：{temp}°C，{desc}"
    else:
        return f"网络请求失败"

# 测试
print(fetch_weather("Beijing"))
print(fetch_weather("Shanghai"))

---

### 3. 异常处理 —— 让程序“优雅降级”

网络抖动、API密钥失效、文件不存在……这些意外在开发中无处不在。异常处理让我们能**主动捕获错误并给出预案**，而不是让程序直接崩溃。

#### 3.1 `try-except` 结构

In [ ]:
import requests

def fetch_data(url):
    try:
        resp = requests.get(url, timeout=5)
        resp.raise_for_status()  # 如果HTTP状态码是4xx或5xx，主动抛出异常
        return resp.json()
    except requests.exceptions.Timeout:
        print(f"错误：请求超时。")
    except requests.exceptions.ConnectionError:
        print(f"错误：网络连接失败，请检查网络。")
    except requests.exceptions.HTTPError as e:
        print(f"错误：服务器响应异常。{e}")
    except Exception as e:
        print(f"发生未知错误：{e}")
    return None

# 测试
data = fetch_data("https://httpbin.org/delay/2") # 一个会延迟2秒响应的测试接口

#### 3.2 手动抛出异常 `raise`
当程序逻辑上出现错误（比如用户输入了不符合要求的值），我们应主动抛出异常。

In [ ]:
def set_temperature(temp):
    if not 0.0 <= temp <= 2.0:
        raise ValueError(f"温度参数 {temp} 不合法，必须在0.0到2.0之间。")
    print(f"温度已设置为 {temp}")

# set_temperature(3.0) # 这会引发 ValueError

try:
    set_temperature(3.0)
except ValueError as e:
    print(e)
except Exception as e:
    print(f"未知错误：{e}")

---

### 4. 正则表达式 —— 文本处理利器（了解）

正则表达式（RegEx）是强大的文本模式匹配工具，在数据清洗和简单文本提取中偶尔用到。这里只做基本了解，**需要时查阅手册即可**。

#### 4.1 最常用的几个元字符和函数
- **函数**: `re.findall()` 返回所有匹配的列表，最常用。
- **字符**: `\d` (数字), `\w` (字母数字下划线), `.` (任意字符), `*` (0次或多次), `+` (1次或多次)。
- **特别提醒**：在 Python 正则表达式字符串前一定要加 `r`，如 `r"\d+"`，避免转义错误。

#### 4.2 代码速览

In [ ]:
# 转义字符串
s = "零基础，学AI，找\"冰可乐\", 不踩坑"
print(s)

# 原生字符串
s = r"零基础，学AI，找\"冰可乐\", 不踩坑"
print(s)


In [ ]:
import re

text = "我的电话是138-0000-1234，邮箱是alice@example.com"

# 查找所有数字
numbers = re.findall(r'\d+', text)
print(numbers)  # ['138', '0000', '1234']

# 提取邮箱
# [\w.+-]+ 逐一解读 
    # []字符集，匹配其中任意一个字符
    # \w 匹配任何单词字符  字母、数字、下划线
    # . + - 就是字面上面的. + -     
    # + 中括号外的+ 是量词 表示匹配前面的字符集一次或多次
# [\w-]+ 逐一解读
    # []字符集 \w单词字符 -  + 两次
# \.[a-zA-Z]{2,}
    # \.    []      {2,}至少匹配2个
emails = re.findall(r'[a-zA-Z0-9_.+-]+@[\w-]+\.[a-zA-Z]{2,}', text)
print(emails)   # ['alice@example.com']

# 替换：隐藏手机号中间四位
hidden_text = re.sub(r'(\d{3})-(\d{4})', r'\1-****', text)
print(hidden_text) # 我的电话是138-****-1234，...

> **学习建议**：除了上面三个例子，遇到更复杂的匹配需求时（如匹配URL、中文等），直接使用搜索引擎查找现成的正则表达式模式即可。

---

### 5. 综合实战：一个健壮的大模型API调用工具

让我们把本节学到的模块、JSON、异常处理知识整合起来，写一个可复用的、能处理错误的简单API调用类。

In [ ]:
# llm_client.py
import json
import requests

class LLMClient:
    def __init__(self, api_key, base_url="https://api.siliconflow.cn/v1"):
        self.api_key = api_key
        self.base_url = base_url

    def chat(self, model, messages):
        """一个简单的聊天请求函数，包含完整的异常处理"""
        url = f"{self.base_url}/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        payload = json.dumps({
            "model": model,
            "messages": messages
        })

        try:
            resp = requests.post(url, headers=headers, data=payload, timeout=10)
            resp.raise_for_status() # 检查HTTP错误
            return resp.json()
        except requests.exceptions.Timeout:
            print("错误：请求大模型超时，请重试。")
        except requests.exceptions.HTTPError as e:
            print(f"错误：API请求失败。状态码: {e.response.status_code}")
            # 可以在这里解析更详细的错误信息
        except Exception as e:
            print(f"发生未知错误：{e}")
        return None

# 测试代码
if __name__ == "__main__":
    # 用一个假的API Key来演示错误处理
    client = LLMClient(api_key="sk-")
    response = client.chat(
        model="Qwen/Qwen2.5-7B-Instruct",
        messages=[{"role": "user", "content": "Hello!"}]
    )
    if response:
        print("请求成功！")
    else:
        print("请求失败，已进行错误处理。")

---

### 6. 优质练习与答案

**练习1 – 模块导入**  
创建包结构 `utils/string_tools.py`，内含函数 `capitalize_words(s)` 将字符串每个单词首字母大写。在 `main.py` 中导入并使用。

In [ ]:
# utils/string_tools.py
def capitalize_words(s):
    return ' '.join(word.capitalize() for word in s.split())

# main.py
from utils.string_tools import capitalize_words
print(capitalize_words("hello world"))  # Hello World

**练习2 – 异常处理**  
编写函数 `divide(a, b)`，返回 a/b。添加 `try-except` 结构，当除数为0时，打印“除数不能为零”并返回 `None`。

In [ ]:
def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("错误：除数不能为零。")
        return None

print(divide(10, 2))  # 5.0
print(divide(10, 0))  # None